# DOAJ Crawler — Artikel Akademik Bahasa Indonesia
**Output:** 3 kolom saja → `judul` · `abstrak` · `kata_kunci`  
**Filter ketat:**
- Abstrak ≥ 50 karakter
- Kata kunci ≥ 3 karakter  
- Abstrak **terdeteksi berbahasa Indonesia** (cek kata-kata khas ID)

Target: **min. 5000 baris bersih**


In [1]:
!pip install requests pandas tqdm -q
print('✓ Library siap.')

✓ Library siap.


In [2]:
import requests
import pandas as pd
import time, re, csv, json
from urllib.parse import urlencode
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
print('✓ Import selesai.')

✓ Import selesai.


In [3]:
# ─────────────────────────────────────────────────────────────
# KONFIGURASI — ubah di sini jika perlu
# ─────────────────────────────────────────────────────────────
OUTPUT_FILE  = 'dataset_id_3kolom.csv'
TARGET_ROWS  = 10000   # target baris bersih (min 5000)
PAGE_SIZE    = 100    # maks 100 per request (batas DOAJ API)
MAX_PAGES    = 100    # per query; 100x100 = 10.000 artikel mentah per query
SLEEP_SEC    = 1.3    # detik antar request (patuhi rate limit DOAJ ~1 req/dtk)

# ─────────────────────────────────────────────────────────────
# DAFTAR QUERY DOAJ
# Urutan: query paling luas dulu, lalu per bidang sebagai cadangan
# Daftar diperluas agar total artikel mentah cukup untuk mencapai
# 5000 baris bersih (rate valid ~30-50%)
# ─────────────────────────────────────────────────────────────
QUERIES = [
    # Query utama — semua jurnal Indonesia + bahasa Indonesia
    'bibjson.journal.country%3AID%20AND%20bibjson.journal.language%3AID',
    # Cadangan per bidang (kalau query utama belum cukup)
    'bibjson.journal.country%3AID%20AND%20pendidikan',
    'bibjson.journal.country%3AID%20AND%20kesehatan',
    'bibjson.journal.country%3AID%20AND%20teknologi%20informasi',
    'bibjson.journal.country%3AID%20AND%20hukum',
    'bibjson.journal.country%3AID%20AND%20ekonomi',
    'bibjson.journal.country%3AID%20AND%20pertanian',
    'bibjson.journal.country%3AID%20AND%20teknik',
    'bibjson.journal.country%3AID%20AND%20sosial%20budaya',
    'bibjson.journal.country%3AID%20AND%20lingkungan%20hidup',
    'bibjson.journal.country%3AID%20AND%20farmasi',
    'bibjson.journal.country%3AID%20AND%20psikologi',
    'bibjson.journal.country%3AID%20AND%20manajemen',
    'bibjson.journal.country%3AID%20AND%20akuntansi',
    'bibjson.journal.country%3AID%20AND%20kebidanan',
    'bibjson.journal.country%3AID%20AND%20keperawatan',
    'bibjson.journal.country%3AID%20AND%20gizi',
    'bibjson.journal.country%3AID%20AND%20komunikasi',
    'bibjson.journal.country%3AID%20AND%20arsitektur',
    'bibjson.journal.country%3AID%20AND%20perikanan',
    'bibjson.journal.country%3AID%20AND%20peternakan',
    'bibjson.journal.country%3AID%20AND%20kehutanan',
    'bibjson.journal.country%3AID%20AND%20matematika',
    'bibjson.journal.country%3AID%20AND%20fisika',
    'bibjson.journal.country%3AID%20AND%20kimia',
    'bibjson.journal.country%3AID%20AND%20biologi',
    'bibjson.journal.country%3AID%20AND%20sastra',
    'bibjson.journal.country%3AID%20AND%20sejarah',
    'bibjson.journal.country%3AID%20AND%20politik',
    'bibjson.journal.country%3AID%20AND%20agama%20islam',
    'bibjson.journal.country%3AID%20AND%20bisnis',
    'bibjson.journal.country%3AID%20AND%20pariwisata',
]

print('✓ Konfigurasi selesai.')
print(f'  Output  : {OUTPUT_FILE}')
print(f'  Target  : {TARGET_ROWS:,} baris bersih')
print(f'  Queries : {len(QUERIES)} query terdaftar')


✓ Konfigurasi selesai.
  Output  : dataset_id_3kolom.csv
  Target  : 10,000 baris bersih
  Queries : 32 query terdaftar


In [4]:
# ─────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────

# Pola kata khas bahasa Indonesia untuk deteksi bahasa
_ID_PATTERN = re.compile(
    r'\b(dan|yang|di|ke|dari|dengan|untuk|pada|dalam|adalah|ini|itu|'
    r'atau|oleh|sebagai|terhadap|antara|pengaruh|analisis|studi|'
    r'penelitian|metode|hasil|kesimpulan|tujuan|abstrak|bahwa|'
    r'menunjukkan|dilakukan|digunakan|dapat|juga|lebih|serta|'
    r'peningkatan|pengembangan|implementasi|berdasarkan|responden|'
    r'variabel|signifikan|efektif|kualitatif|kuantitatif|melalui|'
    r'secara|terdapat|tersebut|diperoleh|tidak|akan|telah|sangat)\b',
    re.IGNORECASE
)

# Pola kata khas bahasa Inggris — untuk menolak abstrak Inggris
# yang kebetulan mengandung 1-2 kata ambigu (mis. 'dan' sbg nama)
_EN_PATTERN = re.compile(
    r'\b(the|this|that|these|those|with|from|research|study|results|'
    r'method|analysis|significant|conclusion|purpose|background|'
    r'objective|showed|using|based|between|approach)\b',
    re.IGNORECASE
)

def clean(text: str) -> str:
    """Hapus whitespace berlebih."""
    if not text:
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip()

def is_indonesian(text: str, min_matches: int = 3) -> bool:
    """
    Deteksi apakah teks berbahasa Indonesia.
    Syarat: minimal `min_matches` kata khas Indonesia ditemukan,
    DAN jumlah kata khas Indonesia harus lebih banyak daripada
    kata khas Inggris (untuk menolak abstrak dwibahasa yang
    didominasi Inggris).
    """
    if not text:
        return False
    id_matches = len(_ID_PATTERN.findall(str(text)))
    en_matches = len(_EN_PATTERN.findall(str(text)))
    return id_matches >= min_matches and id_matches > en_matches

def is_valid_row(judul: str, abstrak: str, kata_kunci: str) -> bool:
    """
    Filter ketat — semua kondisi harus terpenuhi:
    1. Abstrak >= 50 karakter
    2. Kata kunci >= 3 karakter
    3. Abstrak terdeteksi bahasa Indonesia
    """
    ab = clean(abstrak)
    kw = clean(kata_kunci)
    return (
        len(ab) >= 50
        and len(kw) >= 3
        and is_indonesian(ab)
    )

def fetch_page(query: str, page: int) -> dict | None:
    """Ambil satu halaman dari DOAJ API."""
    params = urlencode({'page': page, 'pageSize': PAGE_SIZE})
    url = f'https://doaj.org/api/search/articles/{query}?{params}'
    try:
        r = requests.get(
            url, timeout=30,
            headers={
                'Accept': 'application/json',
                'User-Agent': 'Mozilla/5.0 (academic-research-crawler/2.0)',
            }
        )
        r.raise_for_status()
        return r.json()
    except requests.exceptions.HTTPError as e:
        # DOAJ membatasi page*pageSize <= 10000 -> akan 400 jika kelebihan
        if e.response is not None and e.response.status_code != 400:
            print(f'  ✗ HTTP {e.response.status_code}')
        return None
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return None

def parse_article(hit: dict) -> dict:
    """Ekstrak judul, abstrak, kata_kunci dari satu entri DOAJ."""
    bib = hit.get('bibjson', {})
    judul    = clean(bib.get('title', ''))
    abstrak  = clean(bib.get('abstract', ''))
    kw_list  = bib.get('keywords', []) or []
    kata_kunci = '; '.join(clean(k) for k in kw_list if k)
    return {
        'judul'      : judul,
        'abstrak'    : abstrak,
        'kata_kunci' : kata_kunci,
    }

print('✓ Helper siap.')

✓ Helper siap.


In [5]:
# ─────────────────────────────────────────────────────────────
# CRAWL UTAMA
# ─────────────────────────────────────────────────────────────
data  = []          # hasil bersih
seen  = set()       # deduplikasi via judul (lowercase)
stats = {'req': 0, 'raw': 0, 'skip_lang': 0, 'skip_empty': 0, 'dup': 0}

# DOAJ API membatasi page * pageSize <= 10000
HARD_PAGE_LIMIT = 10000 // PAGE_SIZE

print('='*60)
print(f'  Mulai crawl DOAJ...')
print(f'  Target : {TARGET_ROWS:,} baris bersih')
print('='*60)

for q_idx, query in enumerate(QUERIES):
    if len(data) >= TARGET_ROWS:
        break

    # Cek berapa total artikel tersedia untuk query ini
    probe = fetch_page(query, page=1)
    stats['req'] += 1
    if not probe:
        print(f'\n[Q{q_idx+1}] Gagal probe, skip.')
        continue

    total_avail = probe.get('total', 0)
    max_p = min(MAX_PAGES, HARD_PAGE_LIMIT, (total_avail // PAGE_SIZE) + 1)
    label = query[:55] + '...' if len(query) > 55 else query
    print(f'\n[Q{q_idx+1}/{len(QUERIES)}] {label}')
    print(f'  Tersedia: {total_avail:,} artikel | Akan ambil: {max_p} halaman')
    time.sleep(SLEEP_SEC)

    for page in tqdm(range(1, max_p + 1), desc='  Halaman', unit='hal', leave=False):
        if len(data) >= TARGET_ROWS:
            break

        # Halaman 1 sudah di-fetch saat probe — pakai ulang
        if page == 1:
            result = probe
        else:
            result = fetch_page(query, page)
            stats['req'] += 1

        if not result:
            time.sleep(SLEEP_SEC * 2)
            continue

        hits = result.get('results', [])
        if not hits:
            break

        stats['raw'] += len(hits)
        added = 0

        for hit in hits:
            art = parse_article(hit)
            uid = art['judul'].lower().strip()

            # Deduplikasi
            if not uid or uid in seen:
                stats['dup'] += 1
                continue
            seen.add(uid)

            # Filter bahasa & kelengkapan
            if not is_valid_row(art['judul'], art['abstrak'], art['kata_kunci']):
                # Diagnosa lebih detail
                ab = art['abstrak']
                kw = art['kata_kunci']
                if len(ab) < 50 or len(kw) < 3:
                    stats['skip_empty'] += 1
                else:
                    stats['skip_lang'] += 1
                continue

            data.append(art)
            added += 1

        tqdm.write(
            f'  Hal {page:3d}: {len(hits):3d} mentah '
            f'→ +{added:3d} valid '
            f'→ total {len(data):,}/{TARGET_ROWS}'
        )
        time.sleep(SLEEP_SEC)

# ─── Ringkasan ───────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'  ✓ Selesai!')
print(f'  Total request    : {stats["req"]:,}')
print(f'  Total mentah     : {stats["raw"]:,}')
print(f'  Duplikat dibuang : {stats["dup"]:,}')
print(f'  Dibuang (kosong) : {stats["skip_empty"]:,}')
print(f'  Dibuang (bukan ID): {stats["skip_lang"]:,}')
print(f'  HASIL BERSIH     : {len(data):,} baris')
if len(data) < TARGET_ROWS:
    print(f'  ⚠ Belum mencapai target. Tambahkan lebih banyak QUERIES atau naikkan MAX_PAGES.')
print(f'{"="*60}')


  Mulai crawl DOAJ...
  Target : 10,000 baris bersih

[Q1/32] bibjson.journal.country%3AID%20AND%20bibjson.journal.la...
  Tersedia: 309,799 artikel | Akan ambil: 100 halaman


  Halaman:   0%|          | 0/100 [00:00<?, ?hal/s]

  Hal   1: 100 mentah → + 29 valid → total 29/10000
  Hal   2: 100 mentah → + 23 valid → total 52/10000
  Hal   3: 100 mentah → + 31 valid → total 83/10000
  Hal   4: 100 mentah → + 29 valid → total 112/10000
  Hal   5: 100 mentah → + 23 valid → total 135/10000
  Hal   6: 100 mentah → + 32 valid → total 167/10000
  Hal   7: 100 mentah → + 34 valid → total 201/10000
  Hal   8: 100 mentah → + 31 valid → total 232/10000
  Hal   9: 100 mentah → + 28 valid → total 260/10000
  Hal  10: 100 mentah → + 35 valid → total 295/10000

[Q2/32] bibjson.journal.country%3AID%20AND%20pendidikan
  Tersedia: 88,707 artikel | Akan ambil: 100 halaman


  Halaman:   0%|          | 0/100 [00:00<?, ?hal/s]

  Hal   1: 100 mentah → +  9 valid → total 304/10000
  Hal   2: 100 mentah → +  5 valid → total 309/10000
  Hal   3: 100 mentah → +  8 valid → total 317/10000
  Hal   4: 100 mentah → + 33 valid → total 350/10000
  Hal   5: 100 mentah → + 34 valid → total 384/10000
  Hal   6: 100 mentah → + 37 valid → total 421/10000
  Hal   7: 100 mentah → + 37 valid → total 458/10000
  Hal   8: 100 mentah → + 36 valid → total 494/10000
  Hal   9: 100 mentah → + 29 valid → total 523/10000
  Hal  10: 100 mentah → + 34 valid → total 557/10000

[Q3/32] bibjson.journal.country%3AID%20AND%20kesehatan
  Tersedia: 27,796 artikel | Akan ambil: 100 halaman


...

  Hal   1: 100 mentah → +  1 valid → total 8,989/10000
  Hal   2: 100 mentah → +  7 valid → total 8,996/10000
  Hal   3: 100 mentah → +  8 valid → total 9,004/10000
  Hal   4: 100 mentah → + 12 valid → total 9,016/10000
  Hal   5: 100 mentah → + 11 valid → total 9,027/10000
  Hal   6: 100 mentah → + 14 valid → total 9,041/10000
  Hal   7: 100 mentah → + 10 valid → total 9,051/10000
  Hal   8: 100 mentah → +  8 valid → total 9,059/10000
  Hal   9: 100 mentah → + 11 valid → total 9,070/10000
  Hal  10: 100 mentah → + 10 valid → total 9,080/10000

[Q31/32] bibjson.journal.country%3AID%20AND%20bisnis
  Tersedia: 16,198 artikel | Akan ambil: 100 halaman


  Halaman:   0%|          | 0/100 [00:00<?, ?hal/s]

  Hal   1: 100 mentah → +  0 valid → total 9,080/10000
  Hal   2: 100 mentah → +  3 valid → total 9,083/10000
  Hal   3: 100 mentah → +  3 valid → total 9,086/10000
  Hal   4: 100 mentah → +  2 valid → total 9,088/10000
  Hal   5: 100 mentah → +  6 valid → total 9,094/10000
  Hal   6: 100 mentah → +  6 valid → total 9,100/10000
  Hal   7: 100 mentah → + 11 valid → total 9,111/10000
  Hal   8: 100 mentah → + 19 valid → total 9,130/10000
  Hal   9: 100 mentah → + 22 valid → total 9,152/10000
  Hal  10: 100 mentah → + 17 valid → total 9,169/10000

[Q32/32] bibjson.journal.country%3AID%20AND%20pariwisata
  Tersedia: 3,014 artikel | Akan ambil: 31 halaman


  Halaman:   0%|          | 0/31 [00:00<?, ?hal/s]

  Hal   1: 100 mentah → +  5 valid → total 9,174/10000
  Hal   2: 100 mentah → + 20 valid → total 9,194/10000
  Hal   3: 100 mentah → + 31 valid → total 9,225/10000
  Hal   4: 100 mentah → + 16 valid → total 9,241/10000
  Hal   5: 100 mentah → + 30 valid → total 9,271/10000
  Hal   6: 100 mentah → + 47 valid → total 9,318/10000
  Hal   7: 100 mentah → + 34 valid → total 9,352/10000
  Hal   8: 100 mentah → + 34 valid → total 9,386/10000
  Hal   9: 100 mentah → + 45 valid → total 9,431/10000
  Hal  10: 100 mentah → + 30 valid → total 9,461/10000

  ✓ Selesai!
  Total request    : 2,265
  Total mentah     : 32,000
  Duplikat dibuang : 6,674
  Dibuang (kosong) : 6,175
  Dibuang (bukan ID): 9,690
  HASIL BERSIH     : 9,461 baris
  ⚠ Belum mencapai target. Tambahkan lebih banyak QUERIES atau naikkan MAX_PAGES.


In [6]:
# Preview 5 baris pertama
df_preview = pd.DataFrame(data[:5])
print('Kolom:', df_preview.columns.tolist())
print()
for i, row in df_preview.iterrows():
    print(f'── Baris {i+1} ───────────────────────────────')
    print(f'  Judul     : {row["judul"][:80]}')
    print(f'  Abstrak   : {row["abstrak"][:100]}...')
    print(f'  Kata kunci: {row["kata_kunci"][:80]}')
    print()

Kolom: ['judul', 'abstrak', 'kata_kunci']

── Baris 1 ───────────────────────────────
  Judul     : Analisis Hubungan Penguasaan Kosakata dan Kemampuan Memahami Unsur Intrinsik Cer
  Abstrak   : This research problem is how is the relation between vocabulary mastery and short story intrinsical ...
  Kata kunci: Penguasaan Kosakata; Unsur Intrinsik Cerpen; Siswa SMP; Kendari

── Baris 2 ───────────────────────────────
  Judul     : Evaluasi Penggunaan Aplikasi Sistem Keuangan Desa (Siskeudes) di Nagari Kayu Tan
  Abstrak   : Aplikasi Siskeudes ini merupakan suatu sistem yang dikembangkan oleh BPKP dalam meningkatkan kualita...
  Kata kunci: evaluasi; keuangan desa; siskeudes; sistem informasi manajemen

── Baris 3 ───────────────────────────────
  Judul     : ANALISIS TEKNIS DAN FINANSIAL PRODUKSI ARANG DAN CUKA KAYU DARI LIMBAH INDUSTRI 
  Abstrak   : Limbah serbuk gergaji dan sebetan dari industri penggergajian kayu rakyat secara teknis dapat diguna...
  Kata kunci: Serbuk gergaji; se

In [7]:
# ─────────────────────────────────────────────────────────────
# SIMPAN KE CSV — hanya 3 kolom: judul, abstrak, kata_kunci
# ─────────────────────────────────────────────────────────────
df_out = pd.DataFrame(data, columns=['judul', 'abstrak', 'kata_kunci'])

# Bersihkan nilai kosong
df_out = df_out.fillna('')
df_out = df_out[
    (df_out['judul'] != '') &
    (df_out['abstrak'] != '') &
    (df_out['kata_kunci'] != '')
].reset_index(drop=True)

# Simpan — QUOTE_ALL agar abstrak panjang & koma di dalam teks aman
df_out.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding='utf-8-sig',      # BOM: Excel langsung baca tanpa garbled
    quoting=csv.QUOTE_ALL,     # quote semua field
    lineterminator='\n',
)

import os
size_kb = os.path.getsize(OUTPUT_FILE) / 1024

print(f'✓ Tersimpan: {OUTPUT_FILE}')
print(f'  Baris   : {len(df_out):,}')
print(f'  Kolom   : {list(df_out.columns)}')
print(f'  Ukuran  : {size_kb:.1f} KB')
print()
print('── Statistik Kata Kunci ─────────────────────────────')
# Rata-rata jumlah kata kunci per baris
avg_kw = df_out['kata_kunci'].apply(lambda x: len(x.split(';'))).mean()
print(f'  Rata-rata kata kunci per artikel : {avg_kw:.1f}')
print()
print('── Statistik Panjang Abstrak ────────────────────────')
lens = df_out['abstrak'].str.len()
print(f'  Rata-rata panjang abstrak : {lens.mean():.0f} karakter')
print(f'  Terpendek                 : {lens.min()} karakter')
print(f'  Terpanjang                : {lens.max()} karakter')

✓ Tersimpan: dataset_id_3kolom.csv
  Baris   : 9,461
  Kolom   : ['judul', 'abstrak', 'kata_kunci']
  Ukuran  : 17970.7 KB

── Statistik Kata Kunci ─────────────────────────────
  Rata-rata kata kunci per artikel : 3.3

── Statistik Panjang Abstrak ────────────────────────
  Rata-rata panjang abstrak : 1767 karakter
  Terpendek                 : 67 karakter
  Terpanjang                : 20369 karakter


In [8]:
# Verifikasi akhir — cek tidak ada baris bukan bahasa Indonesia yang lolos
df_check = pd.read_csv(OUTPUT_FILE)
id_count = df_check['abstrak'].apply(is_indonesian).sum()
non_id   = len(df_check) - id_count

print(f'Verifikasi bahasa:')
print(f'  ✓ Bahasa Indonesia : {id_count:,} baris')
print(f'  ✗ Bukan Indonesia  : {non_id:,} baris')

if non_id > 0:
    print()
    print('  Contoh yang bukan Indonesia:')
    mask = ~df_check['abstrak'].apply(is_indonesian)
    print(df_check[mask][['judul','abstrak']].head(3).to_string())
else:
    print()
    print('  Semua baris sudah bahasa Indonesia ✓')

Verifikasi bahasa:
  ✓ Bahasa Indonesia : 9,461 baris
  ✗ Bukan Indonesia  : 0 baris

  Semua baris sudah bahasa Indonesia ✓
